# SPY Vol Surface Calibration

End-to-end pipeline on a real SPY options chain:

1. Load chain from `data/spy_chain_sample.csv`
2. Compute implied vols using our Brent IV solver
3. Fit SVI to the smile for each expiry
4. Fit SSVI jointly across all expiries (no static arbitrage)
5. Plot fitted vs market IV smiles
6. Compute Greeks at each strike
7. Run Heston calibration and overlay the Heston smile

**SPY data**: fetch date 2025-03-14, two expiries (2025-03-21 and 2025-04-17), strikes 80%–115% of spot (S=590).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), ''))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import minimize

from src.pricing.black_scholes import BlackScholes, OptionType
from src.pricing.implied_vol import ImpliedVolSolver, ImpliedVolSolverError
from src.pricing.surface import SVIParams, VolSurface, SSVIVolSurface
from src.pricing.heston import heston_price, heston_implied_vol
from src.pricing.greeks import compute_greeks

plt.style.use('seaborn-v0_8-darkgrid')
COLORS = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']
print('Imports OK')

## 1. Load chain and compute implied vols

In [ ]:
df = pd.read_csv('../data/spy_chain_sample.csv')
print(f'Chain: {len(df)} rows, expiries: {df.expiry.unique()}')

S = float(df.S.iloc[0])
r = 0.053
print(f'S={S}, r={r:.1%}')

# Compute IV using our Brent solver on mid prices
ivs = []
for _, row in df.iterrows():
    ot = OptionType.CALL if row.option_type == 'call' else OptionType.PUT
    try:
        iv = ImpliedVolSolver.solve(row.mid, S, row.strike, row.T, r, ot)
    except ImpliedVolSolverError:
        iv = np.nan
    ivs.append(iv)

df['iv'] = ivs
df = df.dropna(subset=['iv'])
df = df[(df.iv > 0.05) & (df.iv < 1.5)]
print(f'After IV filtering: {len(df)} rows')
df.groupby(['expiry','option_type'])['iv'].describe().round(4)

## 2. SVI calibration — one slice per expiry

In [ ]:
expiries = sorted(df.expiry.unique())
svi_slices = {}

for exp in expiries:
    sub = df[df.expiry == exp].copy()
    T = float(sub.T.iloc[0])
    strikes = sub.strike.values
    market_ivs = sub.iv.values
    svi = VolSurface.fit_slice(market_ivs, strikes, T, S, r)
    svi_slices[exp] = (T, svi)
    print(f'{exp} (T={T:.4f}): a={svi.a:.4f}, b={svi.b:.4f}, rho={svi.rho:.3f}, m={svi.m:.3f}, sigma={svi.sigma:.3f}')

# Build VolSurface object
surface = VolSurface(
    slices={T: svi for exp, (T, svi) in svi_slices.items()},
    S=S, r=r
)
print(f'Calendar spread free: {surface.calendar_spread_check()}')

## 3. SSVI joint calibration

In [ ]:
# Build market vol matrix for SSVI
all_strikes = sorted(df.strike.unique())
all_T = []
mkt_vol_matrix = []

for exp in expiries:
    sub = df[df.expiry == exp].set_index('strike')
    T = float(sub.T.iloc[0])
    row_vols = []
    for K in all_strikes:
        if K in sub.index:
            row_vols.append(float(sub.loc[K, 'iv'] if isinstance(sub.loc[K, 'iv'], float) 
                                  else sub.loc[K, 'iv'].mean()))
        else:
            # Interpolate from SVI
            _, svi = svi_slices[exp]
            F = S * np.exp(r * T)
            k = np.log(K / F)
            row_vols.append(float(svi.implied_vol(k, T)))
    all_T.append(T)
    mkt_vol_matrix.append(row_vols)

mkt_vol_matrix = np.array(mkt_vol_matrix)
all_strikes = np.array(all_strikes)
all_T = np.array(all_T)

ssvi_surface = SSVIVolSurface.fit(mkt_vol_matrix, all_strikes, all_T, S=S, r=r)
p = ssvi_surface.params
print(f'SSVI params: rho={p.rho:.4f}, eta={p.eta:.4f}, gamma={p.gamma:.4f}')
ok, reason = p.is_admissible()
print(f'Admissible: {ok}' + (f' ({reason})' if not ok else ''))
arb = ssvi_surface.is_arbitrage_free()
print('Arbitrage checks:', arb)

## 4. Heston calibration

In [ ]:
# Calibrate Heston to the longer-dated expiry
exp_fit = expiries[-1]
sub = df[df.expiry == exp_fit].copy()
T_fit = float(sub.T.iloc[0])

# Use put IVs for calibration (more liquid, cleaner skew signal)
puts = sub[sub.option_type == 'put'].copy()
K_fit = puts.strike.values
iv_fit = puts.iv.values

def heston_calibration_objective(params):
    v0, kappa, theta, xi, rho = params
    if v0 <= 0 or kappa <= 0 or theta <= 0 or xi <= 0 or abs(rho) >= 1:
        return 1e9
    if 2*kappa*theta <= xi**2:  # Feller
        return 1e9
    err = 0.0
    for K, iv_mkt in zip(K_fit, iv_fit):
        try:
            iv_h = heston_implied_vol(S, K, T_fit, r, v0, kappa, theta, xi, rho, 'put')
            if np.isnan(iv_h): continue
            err += (iv_h - iv_mkt)**2
        except Exception:
            err += 0.01
    return err

# Initial guess: v0=ATM_vol^2, reasonable Heston defaults
atm_iv = float(sub[sub.strike == sub.strike.iloc[(sub.strike - S).abs().argsort().iloc[0]]].iv.mean())
x0 = [atm_iv**2, 2.0, atm_iv**2, 0.3, -0.6]
bounds = [(0.001, 0.5), (0.1, 10.0), (0.001, 0.5), (0.01, 1.5), (-0.99, -0.01)]

print(f'Calibrating Heston to {exp_fit} ({len(K_fit)} put strikes)...')
result = minimize(heston_calibration_objective, x0, method='L-BFGS-B', bounds=bounds,
                   options={'maxiter': 200, 'ftol': 1e-10})
v0_h, kappa_h, theta_h, xi_h, rho_h = result.x
print(f'Heston: v0={v0_h:.4f}(σ₀={np.sqrt(v0_h):.3f}), κ={kappa_h:.3f}, θ={theta_h:.4f}(σ∞={np.sqrt(theta_h):.3f}), ξ={xi_h:.3f}, ρ={rho_h:.3f}')
print(f'Feller: 2κθ={2*kappa_h*theta_h:.4f} > ξ²={xi_h**2:.4f}: {2*kappa_h*theta_h > xi_h**2}')
print(f'Calibration RMSE: {np.sqrt(result.fun/len(K_fit)):.4f} vol pts')

## 5. Plot: Market vs SVI vs SSVI vs Heston

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, exp in zip(axes, expiries):
    sub = df[df.expiry == exp]
    T = float(sub.T.iloc[0])
    F = S * np.exp(r * T)

    # Market IVs
    for ot, marker, label in [('call', 'o', 'Market calls'), ('put', 's', 'Market puts')]:
        s2 = sub[sub.option_type == ot]
        ax.scatter(s2.strike, s2.iv * 100, s=25, marker=marker,
                   label=label, zorder=5, alpha=0.85)

    # Fitted curves on dense strike grid
    K_grid = np.linspace(sub.strike.min() * 0.98, sub.strike.max() * 1.02, 200)
    log_mon = np.log(K_grid / F)

    # SVI
    _, svi = svi_slices[exp]
    iv_svi = svi.implied_vol(log_mon, T) * 100
    ax.plot(K_grid, iv_svi, '-', color='#2196F3', lw=2, label='SVI fit')

    # SSVI
    iv_ssvi = np.array([ssvi_surface.interpolate(K, T) * 100 for K in K_grid])
    ax.plot(K_grid, iv_ssvi, '--', color='#4CAF50', lw=2, label='SSVI fit')

    # Heston smile (only for calibrated expiry)
    if exp == exp_fit:
        iv_heston = []
        for K in K_grid:
            ot_h = 'call' if K >= S else 'put'
            iv_h = heston_implied_vol(S, K, T, r, v0_h, kappa_h, theta_h, xi_h, rho_h, ot_h)
            iv_heston.append(iv_h * 100 if not np.isnan(iv_h) else np.nan)
        ax.plot(K_grid, iv_heston, ':', color='#F44336', lw=2.5, label='Heston fit')

    ax.axvline(S, color='gray', lw=0.8, ls='--', alpha=0.5, label=f'S={S:.0f}')
    ax.set_xlabel('Strike (K)', fontsize=11)
    ax.set_ylabel('Implied Volatility (%)', fontsize=11)
    ax.set_title(f'SPY Vol Smile — {exp}  (T={T:.4f}y)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('SPY Options Chain — SVI / SSVI / Heston Calibration', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../docs/images/vol_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to docs/images/vol_surface.png')

## 6. Greeks at each strike

In [ ]:
exp_show = expiries[-1]
sub = df[(df.expiry == exp_show) & (df.option_type == 'call')].copy()
T = float(sub.T.iloc[0])

greek_rows = []
for _, row in sub.iterrows():
    g = compute_greeks(S, row.strike, T, r, row.iv, OptionType.CALL)
    greek_rows.append({
        'strike': row.strike, 'iv': round(row.iv * 100, 2),
        'price': round(row.mid, 2),
        'delta': round(g.delta, 4),
        'gamma': round(g.gamma, 5),
        'vega':  round(g.vega, 4),
        'theta_day': round(g.theta, 4),
        'vanna': round(g.vanna, 5),
        'volga': round(g.volga, 5),
    })

gdf = pd.DataFrame(greek_rows)
print(f'Greeks for {exp_show} calls (S={S}):')
gdf

## 7. Backtest P&L — gamma hedge vs band hedge

In [ ]:
from src.backtest import BacktestEngine, BacktestConfig
from src.backtest.scenario import ScenarioConfig
from src.hedging.delta_hedger import HedgeFrequency
from src.quoting.market_maker import MMParams
from src.risk.metrics import RiskMetrics

# Use calibrated ATM vol from SPY chain
sigma_spy = float(sub[sub.strike == sub.strike.iloc[(sub.strike - S).abs().argsort().iloc[0]]].iv.mean())
print(f'Using σ={sigma_spy:.3f} (SPY calibrated ATM vol)')

results = {}
for hedge_name, freq in [('Band', HedgeFrequency.BAND), ('Gamma', HedgeFrequency.GAMMA)]:
    cfg = BacktestConfig(
        scenario=ScenarioConfig(S0=S, sigma=sigma_spy, T=0.25, n_steps=63, seed=42),
        strikes_pct=[0.95, 0.975, 1.00, 1.025, 1.05],
        expiry_years=0.25, sigma=sigma_spy, r=r,
        hedge_freq=freq,
        mm_params=MMParams(base_spread_vol=0.005),
        toxicity=0.20,
    )
    res = BacktestEngine(cfg).run()
    results[hedge_name] = res
    pnl = res.records['mtm'].diff().dropna().values
    m = RiskMetrics.summary(pnl, periods_per_year=252)
    print(f'{hedge_name:6s}: PnL={res.total_pnl:+7.2f}  Sharpe={m["sharpe"]:+.3f}  '
          f'MaxDD={m["max_drawdown"]:.2%}  VaR95={m["var_95"]:.3f}  ToxicFills={res.toxic_fill_rate:.1%}')

print('\nSummary:')
print(results['Band'].summary())

## 8. Backtest P&L chart

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# Panel 1: Spot price path
df_band = results['Band'].records
axes[0].plot(df_band['S'], color='#37474F', lw=1.5)
axes[0].set_title(f'SPY Simulated Path  (σ={sigma_spy:.1%}, T=63 steps)', fontweight='bold')
axes[0].set_ylabel('Spot ($)')
axes[0].grid(True, alpha=0.3)

# Panel 2: MtM P&L comparison
for name, color in [('Band', '#2196F3'), ('Gamma', '#F44336')]:
    rec = results[name].records
    mtm = rec['mtm'] - rec['mtm'].iloc[0]
    axes[1].plot(mtm, color=color, lw=1.5, label=f'{name} hedge  (Sharpe={results[name].sharpe:.2f})')
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_title('Mark-to-Market P&L — Band vs Gamma Hedge', fontweight='bold')
axes[1].set_ylabel('P&L ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Panel 3: Net delta with toxic fills marked
axes[2].plot(df_band['net_delta'], color='#FF9800', lw=1.2, label='Net delta (band)')
axes[2].axhline(0, color='black', lw=0.5)
# Mark toxic fill steps
toxic_steps = df_band[df_band['toxic_fills'] > 0]['step']
if len(toxic_steps):
    axes[2].scatter(toxic_steps, df_band.loc[df_band['toxic_fills']>0,'net_delta'],
                    color='#F44336', s=20, zorder=5, label=f'Toxic fills ({len(toxic_steps)} steps)', alpha=0.7)
axes[2].set_title('Net Delta + Toxic Fill Events (Kyle 1985 model)', fontweight='bold')
axes[2].set_ylabel('Net Delta')
axes[2].set_xlabel('Step')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/images/backtest_pnl.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to docs/images/backtest_pnl.png')